## Fase 1. Creación de una bases de datos de películas extraídas de un API

In [1]:
# Hago todas las import que voy a necesitar
import requests                         # para hacer peticion al API
import pandas as pd                     # para poder convertir los datos en dataframe
import mysql.connector 
from mysql.connector import Error
import numpy as np                      #libreria para trabajar con numero para ver si hay numeros nulos o no
import os                               # para guardar contraseña de la coneccion sin que quede visible
from dotenv import load_dotenv
load_dotenv()
password_sql = os.getenv("PASS_SQL")

In [2]:
url_peliculas = "https://beta.adalab.es/resources/apis/pelis/pelis.json"
datos = requests.get(url_peliculas)           #hacemos la coneccion con la libreria

In [3]:
# defino una funcion para saber si el enpoint tiene los datos correctos o no 
def estado_api (endpoint):
    try:
        datos = requests.get(endpoint)
        print(f"Peticion del API {endpoint} con respuesta {datos.status_code}")
        
        if datos.status_code != 200:
            print("peticion del API fallida")
        else:
            return datos.json()
    except:
        print("URL equivocada")

In [4]:
#creo la variable de datos_pelis y guardo ahi la informacion en formato json del estado de la URL_peliculas
datos_pelis = estado_api(url_peliculas)

Peticion del API https://beta.adalab.es/resources/apis/pelis/pelis.json con respuesta 200


In [5]:
# primero veo que tipo de informacion me da el API
type(datos_pelis)

list

In [6]:
# veo como son los datos por dentro y para ello pido informacion de la primera pelicula
datos_pelis[0]

{'id': 1,
 'titulo': 'The Godfather',
 'año': 1972,
 'duracion': 175,
 'genero': 'Crimen',
 'adultos': False,
 'subtitulos': ['es', 'en']}

In [7]:
#convierto el formato json en dataframe
df_peliculas = pd.DataFrame(datos_pelis)


In [8]:
df_peliculas

,id,titulo,año,duracion,genero,adultos,subtitulos
0,1,The Godfather,1972,175,Crimen,False,"[es, en]"
1,2,The Godfather Part II,1974,202,Crimen,False,"[es, en]"
2,3,Pulp Fiction,1994,154,Crimen,True,"[es, en]"
3,4,Forrest Gump,1994,142,Drama,False,"[es, en, fr]"
4,5,The Dark Knight,2008,152,Acción,False,"[es, en]"
...,...,...,...,...,...,...,...
95,96,La vita è bella,1997,116,Drama,False,"[es, en, it]"
96,97,Requiem for a Dream,2000,102,Drama,True,"[es, en]"
97,98,Memento,2000,113,Thriller,True,"[es, en]"
98,99,Eternal Sunshine of the Spotless Mind,2004,108,Drama,False,"[es, en]"


## 2. Creación de la Base de Datos

In [9]:
# 1º me conecto a MySQL
try:
    cnx = mysql.connector.connect(
        host = '127.0.0.1',
        user = 'root',
        password = password_sql
    )
    print('conexion exitosa')
except Error as e:
    print('Error al conectar')

conexion exitosa


In [10]:
# Función para conectarse a MySQL
def conectar_mysql(host="127.0.0.1", user="root", password=password_sql, database=None):
    try:
        cnx = mysql.connector.connect(
            host=host,
            user=user,
            password=password,
            database=database        # si no le pasas base de datos, se conecta al servidor solo
        )
        print("Conexión exitosa")
        return cnx                      # con este return guarda la coneccion y puedo usar esta conexion despues 
    except Error as e:
        print(f"Error al conectar: {e}")

In [11]:
cnx = conectar_mysql()

Conexión exitosa


In [12]:
# cursor es el mensajero va a SQL y te trae la respuesta
cursor = cnx.cursor()

In [13]:
# creacion de la base de datos:
try:
    cursor = cnx.cursor()
    query = 'CREATE DATABASE IF NOT EXISTS peliculas_adalap'
    cursor.execute(query)
    print('Query exitosa')
except Error as e:
    print(e)

Query exitosa


## Fase 3: Inserción de los Datos en la Base de Datos

In [ ]:
# vamos a insertar la informacion a la base de datos creada, para eso miro que tipo de tablas tiene la API
# Creacion tablas
cursor.execute('USE peliculas_adalap')
query = '''CREATE TABLE peliculas( 
    id INT PRIMARY KEY AUTO_INCREMENT,
    titulo VARCHAR(255) NOT NULL,
    anio INT,
    duracion INT,
    genero VARCHAR(100),
    adultos BOOLEAN,
    subtitulos VARCHAR(100)
)'''
cursor.execute(query)

In [15]:
# compruebo si en mi df hay nulos
df_peliculas.isnull().sum() 

id            0
titulo        0
año           0
duracion      0
genero        0
adultos       0
subtitulos    0
dtype: int64

In [16]:
df_peliculas.head()

,id,titulo,año,duracion,genero,adultos,subtitulos
0,1,The Godfather,1972,175,Crimen,False,"[es, en]"
1,2,The Godfather Part II,1974,202,Crimen,False,"[es, en]"
2,3,Pulp Fiction,1994,154,Crimen,True,"[es, en]"
3,4,Forrest Gump,1994,142,Drama,False,"[es, en, fr]"
4,5,The Dark Knight,2008,152,Acción,False,"[es, en]"


In [17]:
# insertar base de datos
query_insert = '''INSERT INTO peliculas (titulo, anio, duracion, genero, adultos, subtitulos)
VALUES (%s, %s, %s, %s, %s, %s)'''

df_peliculas = df_peliculas.rename(columns={'año': 'anio'})
df_peliculas['subtitulos'] = df_peliculas['subtitulos'].astype(str)
columnas_sql = ['titulo', 'anio', 'duracion', 'genero', 'adultos', 'subtitulos']
datos_pelis = df_peliculas[columnas_sql].values.tolist()
cursor.executemany(query_insert, datos_pelis)
cnx.commit()
print(f"Se han insertado nuevos {cursor.rowcount} valores")

Se han insertado nuevos 100 valores


## Fase 4: Obtener información a partir de los datos.
Una vez que tenemos toda la información, vamos a responder las siguientes preguntas utilizando consultas
en SQL:

In [ ]:
# 1. ¿Cuántas películas tienen una duración superior a 120 minutos?

# aqui todavia no he hablado con la base de datos, solo he preparado la pregunta
query_duracion_120 = ''''SELECT                     # creo una variable donde guardar mi consulta de SQL
COUNT(*)                                            # cuenta las filas
FROM peliculas                                      # en la tabla peliculas
WHERE duracion > 120'''                             # solo las que duren mas de 120 minutos

# aqui envio la pregunta con el mensajero cursor y el execute ejecuta la pregunta a MySQL, pero no tengo respuesta
cursor.execute(query_duracion_120)
# fetch = recoge, one= uno, fetchone recoge UNA fila del resltado de la consulta
resultado = cursor.fetchone()

print("Películas > 120 minutos:", resultado[0])

Películas > 120 minutos: 177


In [ ]:
# 1.1.  ¿Cuántas películas tienen una duración superior a 120 minutos? #explicacion de clase
query_duracion = '''SELECT 
COUNT(*) 
FROM peliculas 
WHERE duracion > 120'''

df_duracion = pd.read_sql(query_duracion,cnx)
df_duracion

C:\Users\ana_1\AppData\Local\Temp\ipykernel_41648\1386540548.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_duracion = pd.read_sql(query_duracion,cnx)


,COUNT(*)
0,177


In [39]:
# 2. ¿Cuántas películas incluyen subtítulos en español?

query_sutitulos_español = '''SELECT 
count(*) 
FROM peliculas
WHERE subtitulos 
LIKE "%es%"'''

cursor.execute(query_sutitulos_español)
resultado = cursor.fetchone()

print("Películas subtituladas al español:", resultado[0])

Películas subtituladas al español: 300


In [40]:
# 3. ¿Cuántas películas tienen contenido adulto?

query_peliculas_adulto = '''SELECT 
COUNT(*) 
FROM peliculas 
WHERE adultos 
LIKE "1"'''

cursor.execute(query_peliculas_adulto)
resultado = cursor.fetchone()

print("Películas categorizadas para adultos:", resultado[0])


Películas categorizadas para adultos: 141


In [42]:
# 4. ¿Cuál es la película más antigua registrada en la base de datos?

query_pelicula_antigua = '''select min(anio)
from peliculas'''

cursor.execute(query_pelicula_antigua)
resultado = cursor.fetchone()

print("Película mas antigua registrada:", resultado[0])

Película mas antigua registrada: 1941


In [ ]:
# 5. Muestra el promedio de duración de las películas agrupado por género

query_promedio = '''SELECT genero,
AVG(duracion)
FROM peliculas
GROUP BY genero'''

cursor.execute(query_promedio)
resultado = cursor.fetchall()
 
df_resultado = pd.DataFrame(resultado, columns=['genero', 'promedio_duracion'])
 

df_resultado

In [ ]:
# 6. ¿Cuántas películas por año se han registrado en la base de datos? Ordena de mayor a menor
query_anios= '''SELECT anio, COUNT(*) 
AS total_pelis
FROM peliculas
GROUP BY anio
ORDER BY total_pelis DESC 
'''
cursor.execute(query_anios)
resultado = cursor.fetchall()

df_anio = pd.DataFrame(resultado, columns=['anio','total_pelis'])
df_anio

In [ ]:
# 6.1  ¿Cuántas películas por año se han registrado en la base de datos? Ordena de mayor a menor

query_años = '''SELECT anio, COUNT(*)
FROM peliculas
GROUP BY anio
ORDER BY COUNT(*) DESC'''

cursor.execute(query_anios)

for anio, total in cursor:
    print("Año:", anio, "- Películas:", total)

In [ ]:
# 7. ¿Cuál es el año con más películas en la base de datos?
query_anios= '''SELECT anio, count(*) 
AS total_pelis
FROM peliculas
GROUP BY anio
ORDER BY total_pelis DESC 
LIMIT 1;'''

cursor.execute(query_anios)
resultado = cursor.fetchone()

print(resultado)

(2001, 15)


In [59]:
# 8. Obtén un listado de todos los géneros y cuántas películas corresponden a cada uno
query_generos= '''
SELECT genero, count(*) total_pelis
FROM peliculas
GROUP BY genero
ORDER BY total_pelis desc;'''

cursor.execute(query_generos)
resultado = cursor.fetchall()

df_generos = pd.DataFrame(resultado, columns=['genero', 'peliculas'])


df_generos

,genero,peliculas
0,Drama,81
1,Ciencia ficción,39
2,Acción,27
3,Animación,27
4,Crimen,21
5,Terror,21
6,Thriller,18
7,Fantasía,15
8,Romance,9
9,Aventura,9


In [64]:
# 9. Muestra todas las películas cuyo título contenga la palabra "Godfather" (puedes usar cualquier palabra).

query_godfather = '''
SELECT id, titulo
FROM peliculas
WHERE titulo LIKE '%Godfather%'
'''

cursor.execute(query_godfather)

resultado = cursor.fetchall()

df_godfather = pd.DataFrame(resultado, columns=['id','titulo'])

df_godfather

,id,titulo
0,1,The Godfather
1,2,The Godfather Part II
2,101,The Godfather
3,102,The Godfather Part II
4,201,The Godfather
5,202,The Godfather Part II
